[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BradSegal/OxfordADH2026PythonBootcamp/blob/main/notebooks/08_L7_arrays_and_images.ipynb)

In [ ]:
# Setup: run this cell first. It installs the course helpers (about 20 seconds).
import importlib.util, subprocess, sys

if importlib.util.find_spec("adh_python") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/BradSegal/OxfordADH2026PythonBootcamp.git"],
        check=True,
    )
print("Course helpers ready.")

# Level 7: Libraries, arrays and images

To a computer, a chest X-ray is just a grid of numbers, one number for each
dot (or **pixel**) that says how grey it is. Every AI model that can 'read'
an X-ray starts here. In this level you will open a real X-ray, look at the
numbers that make it up, take some measurements, and get it ready in the way
a neural network expects. The November labs (X-ray classifier and ECG
wearables lab) as well as the MedVision lab all build from here.

**By the end of this level you can:**
- load a library with `import` and use a numpy array instead of a list;
- open an X-ray as an array and explain its shape, its type and what
  the numbers 0 to 255 mean;
- pick out one pixel or a rectangular region, and compare regions;
- show an image and a histogram with matplotlib; and
- turn an X-ray into model input: a fixed size, values from 0 to 1, and
  three channels.

**How the problems work.** As before, write your answer where you see `...`
and then run the cell to check it.

## Libraries

A **library** is code someone else has written, tested, and packaged for you
to use. Python on its own knows about numbers, text, lists, and dicts,
libraries add everything else. Colab already has the ones we need:

- **numpy** (numerical Python) stores numbers in grids called arrays and
  does arithmetic on them quickly;
- **matplotlib** draws charts and images; and
- **Pillow**, whose name in code is `PIL`, opens image files.

`import numpy as np` imports the NumPy library but lets you call it `np` for
short. This is standard for everyone and so any code you look at online will
mirror this. `from PIL import Image` imports only a specific bit of a
library, in this case the bit that lets you work with images. This is
something you've been doing all along with `from adh_python.checks import
...`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from adh_python.checks import level7 as check
from adh_python.data import sample_patient
from adh_python.imaging.images import IMAGES, image_path

## Lists and arrays

The **shock index** is pulse divided by systolic blood pressure. Here are
the readings for our sample patient over a day on the ward. In Level 4
you would work through the two lists with a loop:

In [ ]:
patient = sample_patient()
pulse = patient["observations"]["pulse"]
systolic = patient["observations"]["systolic_bp"]

shock_index = []
for i in range(len(pulse)):
    shock_index.append(round(pulse[i] / systolic[i], 2))
print(shock_index)

A NumPy **array** is a fancy list of numbers but you can do maths to every
element at once which removes the loop. We can convert a list to an array
with `np.array(...)`. Arrays also have their own set of tools that can be
called with a `.` such as `.round(2)` which rounds all the numbers to two
decimal places or `.mean()` which calculates the average without requiring
you to write a loop or `sum(...) / len(...)`.

In [ ]:
pulse_array = np.array(pulse)
systolic_array = np.array(systolic)

shock_index_array = pulse_array / systolic_array
print(shock_index_array.round(2))
print("Mean pulse:", pulse_array.mean())

A trap to see once: `*` means something different for a list. For a list it
repeats the list; for an array it multiplies all the numbers.

In [ ]:
print(pulse * 2)
print(pulse_array * 2)

Arrays have one more power that lists do not. Arrays can have rows **and**
columns. This is exactly what we need for an image.

## A chest X-ray is a grid of numbers

The course package provides five chest X-rays. We can use the `image_path`
function to find a specific X-ray by name then use the `Image.open` function
to load the image and convert it to an array using the `np.array` function.

In [ ]:
print(IMAGES["lobar_pneumonia"]["patient"])
print(IMAGES["lobar_pneumonia"]["findings"])

xray = np.array(Image.open(image_path("lobar_pneumonia")))
print("Shape:", xray.shape)
print("Type of each value:", xray.dtype)
print("Darkest and brightest:", xray.min(), xray.max())

Three facts describe any array:

- **Shape** `(814, 1024)` means 814 rows and 1024 columns. Rows are always
  first and so the shape of an image is **(height, width)**. The round
  brackets are a **tuple** which is like a list that cannot be changed.
- **dtype** (data type) `uint8` is the common type every value in the array
  shares. It is short for *unsigned 8-bit integer* which is a fancy way of
  saying it is a whole number that is never negative and is stored in 8
  bits.
- **min and max** are the darkest and brightest pixels.

Notice how `shape` and `dtype` do not have brackets after them, yet `min()`
and `max()` do. This is because `shape` is a property of the array, i.e.,
it's a stored fact about that array. `max()` is a function, or method, of
the array, i.e., it's a calculation the array performs when asked.

### From bits to grey levels

A **bit** is a single 0 or 1. This means a single bit can encode two things
(black or white). Adding a second bit doubles the amount that can be encoded
to four (00, 01, 10, 11). Each additional bit doubles the amount that can be
stored, so eight bits can encode 2 ** 8 = 256 different values. This means
that if we store a pixel as a `uint8` it can be one of 256 values with 0
being black and 255 being white.

On X-ray, the brightness seen is a result of how much of the beam was
blocked. Bone, metal, and dense soft tissue all absorb X-rays and as such
little of the beam reaches the detector and as such the area appears bright.
Air on the other hand allows full passage of the beam and so aerated lung
appears dark. If this air space is filled with fluid or pus as in a
pneumonia, the lung appears brighter than expected.

Because every value must be between 0 and 255, you cannot have a number
larger than that in a `uint8` array. If you need to use a larger number, you
can create a copy of the array with a larger data type using `.astype(int)`
which uses the standard integer type. You will need to do this later in the
level.

In [ ]:
print(xray.dtype, xray.astype(int).dtype)

## Problem 7.1: Make

The original files behind many digital radiographs store 12 bits per pixel
and not 8. Determine how many levels of grey 12 bits enables and save this
to the variable `levels`. Save the value of the brightest level to the
variable `brightest`. Use the variable `bits` to accomplish this instead of
the value 12.

In [ ]:
bits = 12
levels = ...
brightest = ...
check.grey_levels(bits, levels, brightest)

Sixteen times the amount of shades as our PNG file. A radiologist's viewer
leverages this additional detail when providing the ability to change
brightness and contrast. Images in the teaching files here and in most AI
pipelines utilise 8 bits.

## Seeing the numbers as a picture

`plt.imshow` draws an array as an image. `cmap="gray"` says to draw low
numbers black and high numbers white; without it matplotlib uses a
purple-to-yellow colour scale. `plt.colorbar()` adds a key from grey to
number.

In [ ]:
plt.imshow(xray, cmap="gray")
plt.colorbar()
plt.title("Lobar pneumonia, PA film")
plt.show()

The numbers on the left edge are row numbers and the numbers along the
bottom are column numbers. Row 0 is at the top. As on any PA film, the
patient's right is on the left of the image (the "Sin" marker at top
right marks the patient's left, *sinister*).

A **histogram** counts how many pixels have each grey level.
`xray.ravel()` lays every row end to end in one long line of numbers,
which is what `plt.hist` needs. `bins=64` sorts the grey levels into 64
groups.

In [ ]:
plt.hist(xray.ravel(), bins=64)
plt.xlabel("Grey level (0 = black, 255 = white)")
plt.ylabel("Number of pixels")
plt.title("Histogram of the lobar pneumonia film")
plt.show()

The long, low plateau from ± 70-170 is aerated lung predominantly. The tall
peak in the 230s and 240s is soft tissue, mediastinum, diaphragm, and bone.

## One pixel

Similar to how you indexed a list in Level 4, you can index an array. The
only difference is that you need to specify two positions: `xray[row,
column]`. Arrays start counting from 0, so the last row would be 813 and not
814.

In [ ]:
print("Left upper zone, aerated lung:", xray[250, 750])
print("Mediastinum:", xray[400, 480])
print("Right lower zone, consolidation:", xray[500, 200])

## Slicing a region

A **slice** selects a range of positions, `start:end`. The `start` position
is included, the `end` position is **not** included and the total number of
elements is `end - start`.

In [ ]:
temperatures = patient["observations"]["temperature"]
print(temperatures)
print(temperatures[2:5])  # positions 2, 3 and 4: three readings

On an array of rows and columns, you provide one slice for the rows and one
for the columns separated by a comma: `xray[first_row:end_row,
first_column:end_column]`. This creates a new, smaller array that is a
rectangle cut out of the film.

## Problem 7.2: Predict

**Before you run anything**, predict the shape of this region:

```python
xray[100:300, 50:450]
```

Store your prediction in `prediction` as two numbers in round brackets, rows
first, e.g., `prediction = (10, 20)`. Run the check, then run the code
itself in the cell after.

In [ ]:
prediction = ...
check.predict_crop_shape(prediction)

In [ ]:
print(xray[100:300, 50:450].shape)

## Comparing the lower zones

Radiologists divide each lung into upper, middle and lower **zones**; these
are areas on the film, not lobes. On a PA film the right middle lobe lies
over the right middle and lower zones. This cell cuts out a rectangle of the
right lower zone (on the left of the image) and finds the mean or average
grey level.

In [ ]:
right_lower = xray[450:550, 180:260]
print("Shape:", right_lower.shape)
print(f"Mean grey level: {right_lower.mean():.1f}")

## Problem 7.3: Modify

Now cut out a rectangle of the same size from the left lower zone. The
heart's shadow extends well to the left of the spine, so the lung starts
further out on this side: use the same rows but columns 790 to 870, just
outside the left heart border. Change the code so `left_lower` holds the
left lower zone.

In [ ]:
left_lower = xray[450:550, 180:260]
print(f"Mean grey level: {left_lower.mean():.1f}")
check.left_zone(xray, left_lower)

Now if we compare the two zones side by side. The code for plotting is new,
here's what each bit does:

- `fig, axes = plt.subplots(1, 2, ...)` makes a figure with one row of
  two pictures; `axes` holds the two pictures.
- `figsize=(10, 4)`: Sets the width and height of the figure (in inches).
- `axes[0].imshow(...)` draws on the first picture and `axes[1]` on the
  second. Just like a list, it starts counting from 0.
- `vmin=0` and `vmax=255` ensures that the black is 0 and white is 255 for
  both images. If not set, Matplotlib uses the darkest and lightest pixel in
  each image as black and white and you won't be able to compare them
  accurately.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(right_lower, cmap="gray", vmin=0, vmax=255)
axes[0].set_title(f"Right lower zone, mean {right_lower.mean():.0f}")
axes[1].imshow(left_lower, cmap="gray", vmin=0, vmax=255)
axes[1].set_title(f"Left lower zone, mean {left_lower.mean():.0f}")
plt.show()

The right lower zone is brighter which aligns with consolidation. However, a
mean grey level is not a diagnosis. The value is altered by overlying breast
tissue, a rotated patient, a different exposure, or even a rectangle that
overlaps with the heart shadow. Even normal films differ between sides. If
we look at the `"normal"` film and measure the rectangles `[700:800,
180:260]` (right lower zone) and `[700:800, 680:760]` (left lower zone). The
mean values are 82.1 and 99.5 respectively. If we look at `"normal_2"` and
measure `[560:660, 200:280]` and `[560:660, 720:800]`, the means are 72.0
and 77.8. The left side is brighter in both cases by ± 17 and 6 grey values
respectively. These values are useful for describing the image but not to
make a decision.

## Masks: asking a question of every pixel

In Level 3 a comparison between two numbers gave one `True` or `False`.
A comparison on an array gives a whole array of answers, one per pixel,
called a **mask**. Here the question is "is this pixel darker than
150?", a level chosen by eye for this film to pick out aerated lung.

In [ ]:
level = 150
dark = xray < level
print(dark.dtype, dark.shape)
print(dark[250, 750], dark[400, 480])

plt.imshow(dark, cmap="gray")
plt.title(f"Pixels darker than {level} (white = True)")
plt.show()

The aerated lungs light up. On the patient's right (left of the image)
much of the lower lung is missing from the white area: that is where
consolidation has replaced air.

Python treats `True` as 1 and `False` as 0, so `.sum()` is the number of
`True` pixels. `.size` is the number of pixels in the array. The count over
the size is the **fraction** of pixels that are dark from 0 (none) to 1
(all).

In [ ]:
print(dark.sum(), "of", xray.size, "pixels are darker than", level)
print("Fraction:", dark.sum() / xray.size)

## Problem 7.4: Make

Calculate the fraction of each lower zone that is darker than `level`. Save
the fraction for the right lower zone to `right_fraction` and the fraction
for the left lower zone to `left_fraction`. Use the `right_lower`,
`left_lower`, and `level` variables from above.

In [ ]:
right_fraction = ...
left_fraction = ...
check.dark_fractions(right_lower, left_lower, level, right_fraction, left_fraction)

You should have found about 0.09 for the right lower zone and 0.60 for
the left: far less of the right lower zone looks like aerated lung. In
the four lower-zone rectangles on the normal films given above, every
pixel is darker than 150 (the brightest is 145). A fixed level is still a blunt
tool: it would need changing for a film taken with a different exposure,
and it cannot tell consolidation from a rib. A neural network learns its
own, far richer, tests from thousands of films, but it starts from
exactly these numbers.

## Problem 7.5: Fix the bug

To assist in better viewing lung markings, a colleague has attempted to
brighten the film by adding `amount` to each pixel. This is then thresholded
using `np.minimum(brighter, 255)` which selects the smaller value between
the pixel and 255 (i.e., ensures no value is brighter than white). The code
runs successfully, however, it is clear that the result is incorrect as the
heart, diaphragm, soft tissue around the chest, and several other components
that should be some of the brightest features on the film are dark. Find the
bug and fix the issue.

In [ ]:
amount = 60
brighter = xray + amount
brighter = np.minimum(brighter, 255)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(xray, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")
axes[1].imshow(brighter, cmap="gray", vmin=0, vmax=255)
axes[1].set_title(f"Brightened by {amount}")
plt.show()
check.brightened(xray, amount, brighter)

<details>
<summary><b>Why did that happen?</b> Open this after your own attempt.</summary>

A `uint8` cannot store anything greater than 255 and so NumPy loops back to 0. It's similar to how a 24-hour clock goes from 23:00 to 01:00 when adding
two hours. This meant that a pixel value of 250 when adding 60 became 54, or
almost black. NumPy didn't even warn you! This is known as an **overflow**.
The solution is to convert the image to a higher capacity format using
`.astype(int)` *before* performing the maths. A key lesson is that whenever
adding, subtracting, or multiplying image values, think about if the result
can exceed 0 to 255.

</details>

Even with the bug fixed, everything that was already bright is now pure white,
and its detail is lost. That is why radiology viewers adjust brightness
and contrast together, which radiologists call **windowing**, rather than
simply adding a number.

## Getting an image ready for a model

A neural network is built for one input size, e.g., 224x224 pixels and needs
every image to match. Our five films are all different shapes:

In [ ]:
for name in IMAGES:
    film = np.array(Image.open(image_path(name)))
    print(name, film.shape)

### Smaller images by slicing

A slice can take a third number, a **step**: `start:end:step`. Leaving out
start and end means "from the beginning to the end", so `::4` means "every
4th row" (or column). So if we keep every 4th row but also every 4th column,
we get an image that is 4x smaller in each direction and has 16x fewer
pixels.

In [ ]:
xray_small = xray[::4, ::4]
print(xray.shape, "became", xray_small.shape)

plt.imshow(xray_small, cmap="gray")
plt.title("Every 4th row and column")
plt.show()

There are 204 rows and not 203.5 as it slices the rows from 0, 4, 8, etc.
all the way to 812. The film is still readily identifiable, however, at low
resolution fine details such as fine Kerley B lines of heart failure can
disappear. Determining the size of the input is a game of balancing the
detail you retain as well as the processing power required to use a
particular model.

Slicing by a step doesn't let you get an exact size. In reality pipelines
**resize** the image. Pillow's `resize` function takes in the size as
**(width, height)**, note that this is the opposite to NumPy's shape and is
a common way people end up with squished or stretched images. Squeezing a
rectangular film to a square also squishes it a bit, some pipelines pad or
crop the image to a square first.

In [ ]:
resized = np.array(Image.open(image_path("lobar_pneumonia")).resize((224, 224)))
print(resized.shape)

### Scaling to 0 to 1

Neural networks learn best when their inputs are small numbers on a common
scale. Dividing by 255, the largest value a `uint8` can hold, turns black
into 0.0 and white into 1.0. The result is a **float** (decimal) array. It
is important to always divide by 255 and not the brightest pixel for a given
film so that the same grey shade is the same number for every film.

In [ ]:
xray_scaled = xray_small / 255
print(xray_scaled.dtype, xray_scaled.min(), xray_scaled.max())

### Channels

Most image models were first trained on colour photographs. A colour
pixel is three numbers, how much red, green and blue it has, so a colour
image has shape **(height, width, 3)**. Each of these layers is called a
**channel**. An X-ray has one channel, so to feed it to such a model we
copy it three times. Equal red, green and blue make grey, so it looks
the same.

`np.stack` stacks arrays of the same shape into a larger combined array. The
`axis` parameter controls where this new number is inserted into the shape.
Given that axis 0 refers to the first (row) number of the shape and axis 1
the second (column), using `axis=2` inserts a third number to the shape
after the rows and columns.

In [ ]:
xray_rgb = np.stack([xray_scaled, xray_scaled, xray_scaled], axis=2)
print(xray_rgb.shape)
print("One pixel:", xray_rgb[100, 100])

plt.imshow(xray_rgb)
plt.title("Three identical channels")
plt.show()

PyTorch which you will be using in November, uses channels first: **(3,
height, width)**. The pixels are the same, just the order of the numbers in
the shape is different. If a model complains about the shape of the input,
check where the 3 is first.

## Problem 7.6: Make

To prepare the heart failure film for model input:

1. `small`: every 4th row and every 4th column of `film` (use `step`);
2. `scaled`: `small` with values from 0 to 1; and
3. `model_input`: three copies of `scaled`, shape (height, width, 3).

Every 4th pixel just keeps the example simple, a real pipeline would resize
to a specific size as shown above. The checker goes through each step
sequentially and gives feedback on the first element that needs to be
updated.

In [ ]:
film = np.array(Image.open(image_path("heart_failure")))
step = 4
print("Shape:", film.shape, " Darkest:", film.min(), " Brightest:", film.max())

In [ ]:
small = ...
scaled = ...
model_input = ...
check.prepared_input(film, step, small, scaled, model_input)

It is this that enables an X-ray to be used as an input into a
**convolutional neural network** (CNN) like the November X-ray lab's image
model. The data is formatted as a grid of grey levels, reduced in size (in
reality the image is resized to a fixed size), scaled to be from 0 to 1, and
structured to match the channel order the model expects. The November labs
and MedVision have these steps built in and automatically perform them,
however, you can now appreciate what it is they are doing.

## Before you move on

- An image is an array. Its shape is (height, width): rows come first.
- A slice includes the start and ends just before the end.
- `uint8` only supports values from 0 to 255. If you are going to do
  anything that may result in the number leaving this range, first cast it
  using `.astype(int)`.
- A model requires a fixed size, fixed scale (divide by 255), and the
  channel layout it was trained on.
- Brightness and thresholds are descriptions of pixels, they are not
  diagnoses.

**Next:** Level 8, putting it together: building summaries and prompts
from patient records, the bridge to the MedVision lab.